# Scaled Dot-Product Attention: Step-by-Step Implementation

This notebook walks through the progressive implementation of the attention mechanism, starting from the simplest possible version and gradually adding complexity to approach a more complete transformer-style attention block.

$$
\text{Attention} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{model}}}\right) V
$$

The levels of complexity are described below:
- basic implementation
- \+ batch support
- \+ multi head attention
- \+ masking
- \+ dropout

In [1]:
import numpy as np

In [54]:
# embedding size of 512, 
# total tokens of 10

d_model = 512
seq_len = 10

In [55]:
# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(seq_len, d_model)

## Basic Implementation:
The following implementation provides a **basic, non-batched** version of **scaled dot-product attention**, capturing the essential computation:
- Linear projections for queries ($Q$), keys ($K$), and values ($V$)
- Dot product between $Q$ and $K$
- Scaling by $\sqrt{d_k}$ for numerical stability
- Softmax normalization of attention scores
- Final output as a weighted sum of value vectors

This version is intended to demonstrate the **core mechanics** without additional features like batching, masking, dropout, or multi-head attention. These will be added step-by-step in later sections.


In [56]:
class Attention:
    def __init__(self, d_k):
        self.v = np.random.randn(d_k, d_k)
        self.q = np.random.randn(d_k, d_k)
        self.k = np.random.randn(d_k, d_k)
        self.d_k = d_k

    def softmax(self, x, axis=None):
        x_shifted = x - x.max(axis=axis, keepdims=True) # for numerical stability
        return np.exp(x_shifted) / np.exp(x_shifted).sum(axis=axis, keepdims=True)
    
    def forward(self, X):
        # Linear projections
        Q = X @ self.q
        K = X @ self.k
        V = X @ self.v

        # Scaled dot product
        QK = Q @ K.T
        div_term = self.d_k ** 0.5
        QK_scaled = QK / div_term

        # Attention between tokens
        self.attention = self.softmax(QK_scaled, axis=1)

        # Final projection
        output = self.attention @ V

        return output

    def __call__(self, X):
        return self.forward(X)

In [57]:
attn = Attention(512)

In [58]:
attn(x_input).shape, attn.attention.shape

((10, 512), (10, 10))

Creating this implementation is pretty straightforward as long as we have the formula. One thing that might go wrong is with the softmax implementation. If we simply follow the formula:

$$
\text{softmax}(x) = \frac{\text{exp}(x)}{\Sigma \text{exp(x)}}
$$

We may get `nan` values due to numbers being too large. For example:

In [72]:
def niave_softmax(x):
    return np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)

np.random.seed(1)
x, q, k = np.random.randn(seq_len, d_model), np.random.randn(d_model, d_model), np.random.randn(d_model, d_model)
Q_x, K_x = x @ q, x @ k

print('Q_x mean:\t', Q_x.mean(), '\nQ_x variance:\t' ,Q_x.var())

Q_x mean:	 0.31624628341508076 
Q_x variance:	 515.635402771001


Which is also noted in the paper, "Attention Is All You Need". Furthermore,

In [73]:
scores = Q_x @ K_x.T
print('Q_x mean:\t', scores.mean(), '\nQ_x variance:\t' ,scores.var())

scores = scores / d_model ** 0.5
print('Q_x mean:\t', scores.mean(), '\nQ_x variance:\t' ,scores.var())

Q_x mean:	 -1359.266930498322 
Q_x variance:	 122514913.73370403
Q_x mean:	 -60.0716789998742 
Q_x variance:	 239286.94088614068


Even with dividing the scores by $\sqrt{d_{model}}$, the numbers are still very large:

In [76]:
attention = niave_softmax(scores)

/tmp/ipykernel_6369/953778927.py:2: RuntimeWarning: overflow encountered in exp
  return np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
/tmp/ipykernel_6369/953778927.py:2: RuntimeWarning: invalid value encountered in divide
  return np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)


In [90]:
np.isnan(attention).sum()

np.int64(10)

In [110]:
print(scores[np.isnan(attention)], '\n\n', np.isinf(np.exp(709)), np.isinf(np.exp(710)))

[790.27075501 782.42548704 968.82494697 791.30312033 946.45313136
 922.20983459 938.56788096 832.07557819 754.50898278 771.58655523] 

 False True


/tmp/ipykernel_6369/2899033176.py:1: RuntimeWarning: overflow encountered in exp
  print(scores[np.isnan(attention)], '\n\n', np.isinf(np.exp(709)), np.isinf(np.exp(710)))


In [118]:
# Note that large negative values get approximated as 0 due to the exponential function
np.exp(-23456)

np.float64(0.0)

We see that we run out of space even when working with 64 bit floating points. However, if we work with negative values, then the approximation is 0 instead of infinity. This is beneficial because we can do arithmetic with 0 (where as we cannot with inf). As long as we shift the input to softmax by the same amount for every value in $x$, we obtain the same result as the softmax with the unshifted values.

\begin{align}
\text{softmax} & = \frac{\exp(x-m)}{\Sigma \exp(x - m)}  \\ \\
& = \frac{\exp(x) \cdot \exp(-m)}{\Sigma \exp(x)\cdot\exp(-m)} \\ \\
& = \frac{\exp(-m)\cdot\exp(x)}{\exp(-m) \Sigma \exp(x)} \\ \\
& = \frac{\exp(x)}{\Sigma \exp(x)} \\ \\
& = \text{softmax}
\end{align}

Where we choose $m$ to be the max value in the vector, $x$, so that all values in $x$ are non-positive, ensuring large numbers are approximated as 0 instead of infinity

## Adding Batch Support:

This section extends the basic attention implementation to support **batched input**.

The input tensor now follows the standard shape:

`(batch_size, sequence_length, d_model)`

Key updates include:

- Supporting batch-wise computation of $Q$, $K$, and $V$
- Computing attention scores for each sequence in the batch simultaneously
- Ensuring softmax is applied correctly across the sequence dimension within each batch
- Maintaining output shape consistency: `(batch_size, sequence_length, d_model)`

This implementation is functionally equivalent to the basic implementation but generalized to operate on multiple sequences in parallel.

In [59]:
# embedding size of 512, 
# total tokens of 10

d_model = 512
seq_len = 10
batch_size = 5

In [60]:
# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(batch_size, seq_len, d_model)

In [61]:
tmp_matrix = np.random.randn(d_model, 2)
(x_input @ tmp_matrix).shape

(5, 10, 2)

It seems like python's `@` already supports this, so why do we need to build out a separate implementaiton accounting for batches?

In [62]:
attn = Attention(512)
attn(x_input)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10 is different from 512)

We see this is an error with the Q @ K.T multiplication, what happens is we do `Q @ K.T`. For this to work, 

- Q Should be `[batch_size, input_size, d_model]`
- K Should be `[batch_size, input_size, d_model]`
- K.T Should be `[batch_size, d_model, input_size]`

However, if we take the transpose of a 3d array:

In [63]:
x_input.shape, x_input.T.shape

((5, 10, 512), (512, 10, 5))

We see that the entire order has flipped, we need `batch_size` to stay in the first position for python's matrix multiplication to work

In [64]:
np.transpose(x_input, (0,2,1)).shape

(5, 512, 10)

This worked, let's see how the rest of the function should be affected

In [67]:
tmp_matrix = np.array(
    ([
        [1,1,1],
        [1,2,3],
        [3,3,3],
    ],
    [
        [3,2,1],
        [2,4,6],
        [10,5,1]
    ])
)

print('With axis=1:\n', attn.softmax(tmp_matrix, axis=1))
print()
print('With axis=2:\n', attn.softmax(tmp_matrix, axis=2))

With axis=1:
 [[[1.06506979e-01 9.00305732e-02 6.33789383e-02]
  [1.06506979e-01 2.44728471e-01 4.68310531e-01]
  [7.86986042e-01 6.65240956e-01 4.68310531e-01]]

 [[9.10745952e-04 3.51190270e-02 6.64835448e-03]
  [3.35044712e-04 2.59496460e-01 9.86703291e-01]
  [9.98754209e-01 7.05384513e-01 6.64835448e-03]]]

With axis=2:
 [[[3.33333333e-01 3.33333333e-01 3.33333333e-01]
  [9.00305732e-02 2.44728471e-01 6.65240956e-01]
  [3.33333333e-01 3.33333333e-01 3.33333333e-01]]

 [[6.65240956e-01 2.44728471e-01 9.00305732e-02]
  [1.58762400e-02 1.17310428e-01 8.66813332e-01]
  [9.93185401e-01 6.69203059e-03 1.22568816e-04]]]


We will need to update the axis for softmax, otherwise it will do the columns of the attention matrix for each batch. Since the rowwise sum corresponds to the 3rd dimentions, we need to set `axis=2`.

The rest of the function should be fine as we have no more transposes or dimentions issues

In [162]:
class BatchedAttention:
    def __init__(self, d_k):
        self.v = np.random.randn(d_k, d_k)  # (d_k, d_k)
        self.q = np.random.randn(d_k, d_k)  # (d_k, d_k)
        self.k = np.random.randn(d_k, d_k)  # (d_k, d_k)
        self.d_k = d_k

    def softmax(self, x, axis=None):
        x_shifted = x - x.max(axis=axis, keepdims=True)  # for numerical stability
        return np.exp(x_shifted) / np.exp(x_shifted).sum(axis=axis, keepdims=True)

    def forward(self, X):
        # X: (batch_size, seq_len, d_k)

        # Linear projections
        Q = X @ self.q  # [batch_size, seq_len, d_k]
        K = X @ self.k  # [batch_size, seq_len, d_k]
        V = X @ self.v  # [batch_size, seq_len, d_k]

        # Scaled dot product
        QK = Q @ np.transpose(K, (0, 2, 1))  # [batch_size, seq_len, seq_len]
        div_term = self.d_k ** 0.5
        QK_scaled = QK / div_term  # [batch_size, seq_len, seq_len]

        # Attention weights
        self.attention = self.softmax(QK_scaled, axis=2)  # [batch_size, seq_len, seq_len]

        # Apply attention to values
        output = self.attention @ V  # [batch_size, seq_len, d_k]

        return output

    def __call__(self, X):
        return self.forward(X)


In [163]:
attn = BatchedAttention(512)
attn(x_input).shape

(5, 10, 512)

It worked, let's pass the same input as two different batches to see if we get the same output

In [70]:
x_input = np.random.randn(seq_len, d_model)
x_input = np.stack((x_input, x_input))
x_input.shape

(2, 10, 512)

In [71]:
output = attn(x_input)
print((output[0] == output[1]).all())

True


## Implementing Multi-Head Attention:

In this section, we enhance the attention mechanism by implementing **multi-head attention**, which allows the model to learn many different patterns *(similar to having multiple filters in a convolutional neural network)*.

Key concepts introduced here:

- Splitting the input projections ($Q$, $K$, and $V$) into multiple heads along the feature dimension.
- Performing scaled dot-product attention independently for each head in parallel.
- Concatenating the outputs of all heads back into a single tensor.
- Applying a final linear projection to combine the multi-head outputs into the original feature dimension.

Multi-head attention increases the model’s ability to capture diverse aspects of the input sequences and has become a standard building block in transformer architectures.

This implementation assumes batched inputs and builds directly on the batching functionality added in the previous section.

In [112]:
d_model = 512
seq_len = 10
batch_size = 5
h = 8 # Number of heads
d_k = d_model // h # Needs to be an int (i.e., d_model needs to be a multiple of h)

# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(batch_size, seq_len, d_model)

In [164]:
q, k, v = np.random.randn(d_model, d_model), np.random.randn(d_model, d_model), np.random.randn(d_model, d_model)
o = np.random.randn(d_model, d_model)

In [165]:
q.shape, x_input.shape

((512, 512), (5, 10, 512))

python multiplication cannot multiply two matricies with different batch sizes (e.g., the 'batch size' for multihead attention is $h=8$, and the batch size for our data is 5).

To multiply these tensors, we will need to find a proper way to reshape.

I got stumped here. The idea is to use blockwise matrix multiplication.

- Since each head will only multiply by a section of the input embeddings, it is rather unintuitive that we can start similar as before, with $Q_w = XQ$
- However, since we know blockwise matrix multiplication rules work, we can then separate out seperate sections (heads) after multiplying
- And finally, to maintaing neat matrix multiplications between $Q$, $K$, and $V$, we combine each head with the batchs 

The idea was taken from this lovely repo: [models-from-scratch-python](https://github.com/cristianleoo/models-from-scratch-python/blob/main/Multi-Head%20Attention/demo.py)

If we didn't do it this way, then we'd need to init each q, k, v matrix with size $d_k \times d_k$ instead of what we did ($d_{model} \times d_{model}$).

Below is an example

In [166]:
print(x_input.shape, q.shape)
Q = x_input @ q # Do matrix multiplication
print(Q.shape)
Q = Q.reshape(Q.shape[0], Q.shape[1], h, -1) # Split the multiplication into 8 heads (blockwise mat mult)
print(Q.shape)
Q = Q.transpose(0, 2, 1, 3) # Move the heads next to the batches so we can stack them
print(Q.shape)
Q = Q.reshape(-1, Q.shape[2], Q.shape[3]) # Combine batches with the attention heads
print(Q.shape)

(5, 10, 512) (512, 512)
(5, 10, 512)
(5, 10, 8, 64)
(5, 8, 10, 64)
(40, 10, 64)


In [167]:
class MultiheadAttention:
    def __init__(self, d_model, h):
        self.v = np.random.randn(d_model, d_model)
        self.q = np.random.randn(d_model, d_model) 
        self.k = np.random.randn(d_model, d_model) 
        self.o = np.random.randn(d_model, d_model)
        
        self.d_model = d_model
        self.h = h 
        self.d_k = d_model // h

    def softmax(self, x, axis=None):
        x_shifted = x - x.max(axis=axis, keepdims=True)  # for numerical stability
        return np.exp(x_shifted) / np.exp(x_shifted).sum(axis=axis, keepdims=True)

    def multiply_qkv(self, input_data, x):
        # Do matrix multiplication
        X = input_data @ x                                             # [batch_size, seq_len, d_model]

        # Split the multiplication into 8 heads (blockwise mat mult)
        X = X.reshape(X.shape[0], X.shape[1], self.h, -1)              # [batch_size, seq_len, heads, d_k]

        # Move the heads next to the batches so we can stack them
        X = X.transpose(0, 2, 1, 3)                                    # [batch_size, heads, seq_len, d_k]
        
        # Combine batches with the attention heads
        X = X.reshape(-1, X.shape[2], X.shape[3])                      # [batch_size * heads, seq_len, d_k]
        
        return X

    def concat(self, X):
        # Reverse the previous steps from multiply_qkv
        X = X.reshape(-1, self.h, X.shape[1], X.shape[2])         # [batch_size, heads, seq_len, d_k]
        X = X.transpose(0, 2, 1, 3)                               # [batch_size, seq_len, heads, d_k]
        X = X.reshape(X.shape[0], X.shape[1], -1)                 # [batch_size, seq_len, d_model]
        return X

    def forward(self, X):
        # X: (batch_size, seq_len, d_k)

        # Linear projections
        Q = self.multiply_qkv(X, self.q) # [batch_size * h, seq_len, d_k]
        K = self.multiply_qkv(X, self.k) # [batch_size * h, seq_len, d_k]
        V = self.multiply_qkv(X, self.v) # [batch_size * h, seq_len, d_k]


        # Scaled dot product
        scores = Q @ K.transpose(0,2,1) / Q.shape[-1] ** 0.5  # [batch_size * h, seq_len, seq_len]

        # Attention weights
        self.attention = self.softmax(scores, axis=-1)  # [batch_size * h, seq_len, seq_len]

        # Apply attention to values
        multi_head_attention = self.attention @ V  # [batch_size * h, seq_len, d_k]

        # Concatenate heads (reverse previous manipulation)        
        multi_head_attention = self.concat(multi_head_attention)

        output = (multi_head_attention @ self.o)
        
        return output

    def __call__(self, X):
        return self.forward(X)


In [168]:
attn = MultiheadAttention(d_model, h)

In [169]:
attn(x_input).shape

(5, 10, 512)

## Incorporating Masking into Attention

In this section, we enhance our attention mechanism by adding **masking**.

Masking allows the model to:

- Ignore padding tokens in variable-length sequences (padding mask)
- Prevent attending to future tokens in autoregressive tasks like text generation (causal or look-ahead mask)

This is useful because it previents the attention scores attending to future words in a text. This would be bad as these are autoregressive models and future tokens affecting the weights of the current token would not apply to test time.


#### Key Modifications:

The attention scores (before softmax) are modified by the mask:

- $scores = \frac{QK^T}{\sqrt{d_k}} + mask$
- The mask is typically implemented as an additive mask where
  - Tokens to keep have a value of 0
  - Tokens to mask have a large negative value (e.g., -1e9). This makes it so they will get a value of zero after softmax

#### Functional Impact:

 - The softmax operation ensures that masked positions receive zero attention weight
 - The model learns to attend only to relevant positions in each head and timestep

This implementation continues to support multi-head attention and batching, and simply adds an optional mask argument to the attention forward pass.

In [2]:
d_model = 512
seq_len = 10
batch_size = 5
h = 8 # Number of heads
d_k = d_model // h # Needs to be an int (i.e., d_model needs to be a multiple of h)

# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(batch_size, seq_len, d_model)

In [3]:
q, k, v = np.random.randn(d_model, d_model), np.random.randn(d_model, d_model), np.random.randn(d_model, d_model)
o = np.random.randn(d_model, d_model)

In [4]:
q.shape, x_input.shape

((512, 512), (5, 10, 512))

The difficult part of this implementation will be getting the broadcasting to line up correctly

## Adding Dropout to Attention

In this section, we integrate **dropout** into the attention mechanism to improve generalization and reduce overfitting during training.

In attention mechanisms, dropout is typically applied to the attention weights after softmax but before applying them to the values:

$$
\text{Attention}(Q, K, V) = \text{Dropout}\left(\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right)\right) V
$$

This encourages the model to not overly rely on specific positions, leading to more robust attention patterns.

#### Key Modifications:

- After computing attention weights via softmax, we apply dropout:
   - ```python
     attention_weights = self.dropout(self.softmax(scores))
     ```
</br> 
- The dropout layer randomly zeros out elements with a specified dropout probability (e.g., 0.1), which is a hyperparameter.